# Gold Layer Overview

This notebook implements the **Gold layer** — the final stage of the Medallion Architecture (Bronze → Silver → Gold). Bronze landed the raw Olist data, Silver cleaned and standardized it, and Gold now reshapes it into **business-oriented analytical tables**: one row per business entity (order, product, customer, seller, category, state) or time period, with metrics pre-aggregated for reporting.

How Silver becomes Gold:

* The five Silver tables (`silver_orders`, `silver_order_items`, `silver_products`, `silver_customers`, `silver_sellers`) are the only inputs, loaded with `spark.table(...)`.
* **Joins** relate the cleaned entities to each other — orders to their items, items to their products and sellers, customers to their order analytics — so business questions spanning multiple datasets can be answered.
* **Aggregations** (`groupBy` with `count` / `sum` / `avg`) condense millions of item-level rows into revenue, volume, delivery and customer-behavior metrics.

Joins and aggregations are where business insight is created: delivery and cancellation rates, monthly trends, top product categories, customer spending and seller performance only exist once the entities are combined and summarized. The result is a set of Delta tables in `workspace.olist` that dashboards and reports can query directly.

In [0]:
# Dependency cell (added during notebook split).
# Imports used by the cells below — mirrors the wildcard import in the original
# olist_full_backup notebook (cell 14) so this notebook runs standalone.
from pyspark.sql.functions import *

In [0]:
# Dependency cell (added during notebook split).
# Load the Silver Delta tables produced by 02_silver_transformations using spark.table().
silver_orders = spark.table('workspace.olist.silver_orders')
silver_order_items = spark.table('workspace.olist.silver_order_items')
silver_products = spark.table('workspace.olist.silver_products')
silver_customers = spark.table('workspace.olist.silver_customers')
silver_sellers = spark.table('workspace.olist.silver_sellers')

# GOLD orders

## Order Analytics

Order analytics starts from `gold_orders` — a column selection from `workspace.olist.silver_orders` (`order_id`, `order_status`, `customer_id`, the timestamp columns, `is_delivered`, `delivery_days` and `order_estimated_delivery_date`).

**Order-level KPIs:**

* `Total_order_placed`, `Total_order_delivered` and the delivered percentage — how much of the order volume is actually fulfilled.
* Orders per `order_status` and the `Cancellation_rate` — the share of orders that were canceled.
* `Average_delivery_days` — mean of `delivery_days` over delivered orders.
* `Late_delivery_rate` — delivered orders where `order_delivered_customer_date` exceeds `order_estimated_delivery_date`, as a share of delivered orders.
* **Monthly metrics** — orders per `order_year` / `order_month` (persisted as `workspace.olist.monthly_orders`), plus average delivery days and late-delivery rate per month.

**Combining orders with item summaries:** `silver_orders` is joined to `silver_order_items` on `order_id` (inner join), and the joined rows are aggregated per order into `golden_order_summary`:

* `total_items` — `count(order_item_id)`, how many items the order contains
* `total_price` — `sum(price)`, the order's product value
* `total_freight_value` — `sum(freight_value)`, the order's shipping cost
* `total_item_value` — `sum(item_value)`, product value plus freight

A `left_anti` join identifies orders that exist in `silver_orders` but have no item records — inspected by `order_status` — making it clear which statuses those unmatched orders belong to.

**`gold_order_analytics`** joins `silver_orders` with `golden_order_summary` using a **LEFT JOIN on `order_id`**. The left join retains every order, including orders with no matching item records — their item-summary columns are simply null. Order-level metrics such as delivery and cancellation rates must count all orders; an inner join would silently drop the item-less orders and understate order counts. The result is persisted as `workspace.olist.gold_order_analytics`.

In [0]:
gold_orders=spark.sql('select order_id,order_status,customer_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,             order_delivered_customer_date,is_delivered,delivery_days ,order_estimated_delivery_date from workspace.olist.silver_orders')

In [0]:
# Total number of order placed and how many of them we are able to deliver
new_gold=gold_orders.select(
    count("order_id").alias("Total_order_placed"),
    sum(
        when(col("is_delivered") == "yes", 1)
        .otherwise(0)
    ).alias("Total_order_delivered")
    
)
new_gold.display()
# Percentage of order delivered
new_gold.select(
    (col('Total_order_delivered')/col('Total_order_placed')*100).alias("Percentage of order delivered")
).display()

In [0]:
# No of orders according to each order status
new_gold1=gold_orders.groupBy("order_status").agg(count("order_id").alias("Total_orders")).orderBy(col('Total_orders').desc())
new_gold1.display()

In [0]:
# Cancellation rate of Orders
new_gold1.select(
    round((sum(when(col('order_status')=='canceled',col('Total_orders')))
    /sum(col('Total_orders'))*100),2).
    
    alias('Cancellation_rate')
).display()

In [0]:
# avg delivery days
gold_orders.filter((col('is_delivered')=='yes')&(col('delivery_days').isNotNull())).select(round(avg(col('delivery_days')),2).alias('Average_delivery_days')).display()

In [0]:
# Late order delivery rate
gold_orders.select(
    round(sum(
        when((col('is_delivered')=='yes')&(col('order_delivered_customer_date')>col('order_estimated_delivery_date')),1).
        otherwise(0)
    )/
    sum(
        when((col('is_delivered')=='yes') & (col('delivery_days').isNotNull()),1).
        otherwise(0)
    )*
    100,2).
    alias('Late_delivery_rate')
).display()

In [0]:
# No of orders per month
monthly_orders = gold_orders \
    .withColumn("order_year", year(col("order_purchase_timestamp"))) \
    .withColumn("order_month", month(col("order_purchase_timestamp"))) \
    .groupBy("order_year", "order_month") \
    .agg(count("order_id").alias("Total_orders")) \
    .orderBy("order_year", "order_month")

monthly_orders.display()

In [0]:
monthly_orders.write.format('delta').mode('overwrite').saveAsTable('workspace.olist.monthly_orders')

In [0]:
(gold_orders.filter((col('is_delivered')=='yes')&(col('delivery_days').isNotNull()))
.withColumn("order_year", year(col("order_purchase_timestamp")))
.withColumn("order_month", month(col("order_purchase_timestamp")))
.groupBy('order_year','order_month')
.agg(round(avg(col('delivery_days')),2).alias('average_delivery_days'))
.orderBy('order_year','order_month').display() )

In [0]:
(
    gold_orders
    .filter(
        (col('is_delivered') == 'yes') &
        (col('delivery_days').isNotNull())
    )
    .withColumn("order_year", year(col("order_purchase_timestamp")))
    .withColumn("order_month", month(col("order_purchase_timestamp")))
    .groupBy("order_year", "order_month")
    .agg(
        round((
            sum(
                when(
                    col('order_delivered_customer_date') >
                    col('order_estimated_delivery_date'),
                    1
                ).otherwise(0)
            )
            /
            sum(
                when(
                    col('order_delivered_customer_date').isNotNull() &
                    col('order_estimated_delivery_date').isNotNull(),
                    1
                ).otherwise(0)
            )
            * 100
        ),2).alias("Late_delivery")
    )
    .orderBy("order_year", "order_month")
    .display()
)

In [0]:
# loading both silver tables
silver_orders = spark.table("workspace.olist.silver_orders")
silver_order_items = spark.table("workspace.olist.silver_order_items")

In [0]:
orders_with_items=silver_orders.join(silver_order_items, on='order_id', how='inner')


In [0]:
orders_with_items.display()

In [0]:
orders_with_items.count()

In [0]:
orders_with_items.groupBy('order_id').agg(
    count(col('order_item_id')).alias('total_items'),
    sum(col('price')).alias('total_price'),
    sum(col('freight_value')).alias('total_freight_value'),
    sum(col('item_value')).alias('total_item_value')
).display()

# Golden order_items

In [0]:
golden_order_summary=orders_with_items.groupBy('order_id').agg(
    count(col('order_item_id')).alias('total_items'),
    sum(col('price')).alias('total_price'),
    sum(col('freight_value')).alias('total_freight_value'),
    sum(col('item_value')).alias('total_item_value')
)

In [0]:
golden_order_summary.count()

In [0]:
golden_order_summary.select(col('order_id')).distinct().count()

In [0]:
golden_order_summary.agg(
    sum(
        when(col('order_id').isNull(), 1).otherwise(0)
    ).alias('null_order_id'),
    sum(
        when(col('total_items').isNull(), 1).otherwise(0)
    ).alias('null_total_items'),
    sum(
        when(col('total_price').isNull(), 1).otherwise(0)
    ).alias('null_total_price'),
    sum(
        when(col('total_freight_value').isNull(), 1).otherwise(0)
    ).alias('null_total_freight_value'),
    sum(
        when(col('total_item_value').isNull(), 1).otherwise(0)
    ).alias('null_total_item_value')

        ).display()


In [0]:
# Inspecting the values that are present in silver_orders but not in silver_order_items
silver_orders.join(silver_order_items,on='order_id',how="left_anti").display()

In [0]:
silver_orders.join(silver_order_items,on='order_id',how="left_anti").groupBy(col('order_status')).count().display()


In [0]:
# Creating golden_order_items table
golden_order_summary.write.mode('overwrite').saveAsTable('workspace.olist.golden_order_summary')

In [0]:
golden_order_summary.count()


In [0]:
%sql
DESCRIBE DETAIL workspace.olist.golden_order_summary


In [0]:
# Gold order analytics
gold_order_analytics=silver_orders.join(golden_order_summary,on='order_id',how='left')

In [0]:
gold_order_analytics.display()
gold_order_analytics.count()


In [0]:
gold_order_analytics.distinct().count()

In [0]:
gold_order_analytics.agg(
    sum(when(col('total_items').isNull(),1).otherwise(0)).alias('null_items'),
    sum(when(col('total_item_value').isNull(),1).otherwise(0)).alias('null_order_value')
).display()

In [0]:
# creating gol order analytics table
gold_order_analytics.write.format('delta').mode('overwrite').saveAsTable('workspace.olist.gold_order_analytics')

In [0]:
%sql
describe detail workspace.olist.gold_order_analytics

## Product and Category Analytics

Product analytics joins `silver_products` to `silver_order_items` on `product_id` (inner join) into `products_with_order_items`, connecting each sold item row to its product.

**Per-product performance** (`gold_product_performance`, grouped by `product_id`):

* `total_units_sold` — `count(order_id)`, how many units of the product were sold
* `total_revenue` — `sum(price)`, revenue from the product
* `total_freight` — `sum(freight_value)`, shipping charged on those units
* `total_order_value` — `sum(price) + sum(freight_value)`, the full monetary value
* `avg_selling_price` — `avg(price)`, the average unit price

A validation aggregate re-checks these figures, including a `calculation_mismatches` count comparing `total_revenue + total_freight` against `total_order_value`. The product category is attached with a LEFT JOIN against `silver_products` (`product_category_name`), producing `gold_product_analytics` (`workspace.olist.gold_product_analytics`).

**Category performance** (`gold_category_performance`, grouped by `product_category_name`): `total_products` (`countDistinct(product_id)`), summed units, revenue, freight and order value, plus a category-level `avg_selling_price` computed as `sum(total_revenue) / sum(total_units_sold)`. Top-10 lists by revenue, units sold and average selling price identify the highest-earning, best-selling and premium-priced categories — the categories the business should stock deeper, promote, or price strategically. Persisted as `workspace.olist.gold_category_performance`.

In [0]:
products_with_order_items=silver_products.join(silver_order_items,on='product_id',how='inner')

In [0]:
products_with_order_items.display()

In [0]:
products_with_order_items.count()


In [0]:
silver_order_items.count()

In [0]:
products_with_order_items.select('product_id').distinct().count()

# Gold Products

In [0]:
gold_product_performance = products_with_order_items.groupBy('product_id').agg(
    count('order_id').alias('total_units_sold'),
    round(sum('price'), 2).alias('total_revenue'),
    round(sum('freight_value'), 2).alias('total_freight'),
    round(sum('price') + sum('freight_value'), 2).alias('total_order_value'),
    round(avg('price'), 2).alias('avg_selling_price')
)

In [0]:
gold_product_performance.display()

In [0]:
# validation
gold_product_performance.agg(
    count('*').alias('total_rows'),
    countDistinct('product_id').alias('distinct_products'),
    sum(when(col('total_units_sold').isNull(), 1).otherwise(0)).alias('null_units'),
    sum(when(col('total_revenue').isNull(), 1).otherwise(0)).alias('null_revenue'),
    sum(when(col('total_freight').isNull(), 1).otherwise(0)).alias('null_freight'),
    sum(when(col('total_order_value').isNull(), 1).otherwise(0)).alias('null_order_value'),
    sum(when(col('avg_selling_price').isNull(), 1).otherwise(0)).alias('null_avg_price'),
    sum(
        when(
            round(col('total_revenue') + col('total_freight'), 2) != col('total_order_value'),
            1
        ).otherwise(0)
    ).alias('calculation_mismatches')
).display()


In [0]:
gold_product_analytics = gold_product_performance.join(
    silver_products.select('product_id', 'product_category_name'),
    on='product_id',
    how='left'
).select(
    'product_id',
    'product_category_name',
    'total_units_sold',
    'total_revenue',
    'total_freight',
    'total_order_value',
    'avg_selling_price'
)

gold_product_analytics.display()

In [0]:
gold_product_analytics.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.gold_product_analytics')

In [0]:
%sql
DESCRIBE DETAIL workspace.olist.gold_product_analytics;

In [0]:
gold_category_performance = gold_product_analytics.groupBy(
    'product_category_name'
).agg(
    countDistinct('product_id').alias('total_products'),
    sum('total_units_sold').alias('total_units_sold'),
    round(sum('total_revenue'), 2).alias('total_revenue'),
    round(sum('total_freight'), 2).alias('total_freight'),
    round(sum('total_order_value'), 2).alias('total_order_value'),
    round(
        sum('total_revenue') / sum('total_units_sold'), 2
    ).alias('avg_selling_price')
)

gold_category_performance.display()

In [0]:
gold_category_performance.orderBy(
    col('total_revenue').desc()
).limit(10).display()

In [0]:
gold_category_performance.orderBy(
    col('total_units_sold').desc()
).limit(10).display()

In [0]:
gold_category_performance.orderBy(
    col('avg_selling_price').desc()
).limit(10).display()

## Customer Analytics

Customer analytics joins `silver_customers` to `gold_order_analytics` on `customer_id` with a LEFT JOIN — every customer row is retained, with each order's analytics attached.

**Why `customer_unique_id`:** in the Olist schema `customer_id` is unique per order — the same person returning for another order receives a new `customer_id`. Aggregating by `customer_id` would count one repeat buyer as many separate customers. `customer_unique_id` identifies the underlying person, so grouping by it produces true customer-level metrics.

**Metrics** (`gold_customer_performance`, grouped by `customer_unique_id`, final version):

* `total_orders` — `countDistinct(order_id)`, how many orders the person placed
* `orders_with_item_data` — `count(total_item_value)`, orders that have item records
* `orders_missing_item_data` — orders whose item summary is null (orders without matching order items)
* `total_spending` — `sum(total_price)`: based on **product price only, excluding freight**
* `avg_order_value` — `avg(total_item_value)`: uses `total_item_value`, so it **includes freight**
* `total_items_purchased` — `sum(total_items)`, total items bought across orders

The first version of this aggregation counted only orders, spending, average order value and items; after inspecting the orders with null `total_item_value` by `order_status`, the `orders_with_item_data` / `orders_missing_item_data` columns were added so item-less orders are visible rather than hidden. Persisted as `workspace.olist.gold_customer_analytics`.

In [0]:
gold_category_performance.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.gold_category_performance')

## Customer Analytics

Customer analytics joins `silver_customers` to `gold_order_analytics` on `customer_id` with a LEFT JOIN — every customer row is retained, with each order's analytics attached.

**Why `customer_unique_id`:** in the Olist schema `customer_id` is unique per order — the same person returning for another order receives a new `customer_id`. Aggregating by `customer_id` would count one repeat buyer as many separate customers. `customer_unique_id` identifies the underlying person, so grouping by it produces true customer-level metrics.

**Metrics** (`gold_customer_performance`, grouped by `customer_unique_id`, final version):

* `total_orders` — `countDistinct(order_id)`, how many orders the person placed
* `orders_with_item_data` — `count(total_item_value)`, orders that have item records
* `orders_missing_item_data` — orders whose item summary is null (orders without matching order items)
* `total_spending` — `sum(total_price)`: based on **product price only, excluding freight**
* `avg_order_value` — `avg(total_item_value)`: uses `total_item_value`, so it **includes freight**
* `total_items_purchased` — `sum(total_items)`, total items bought across orders

The first version of this aggregation counted only orders, spending, average order value and items; after inspecting the orders with null `total_item_value` by `order_status`, the `orders_with_item_data` / `orders_missing_item_data` columns were added so item-less orders are visible rather than hidden. Persisted as `workspace.olist.gold_customer_analytics`.

In [0]:
customer_order_details = silver_customers.join(
    gold_order_analytics,
    on='customer_id',
    how='left'
)

customer_order_details.display()

In [0]:
customer_order_details.agg(
    count('*').alias('total_rows'),
    countDistinct('customer_id').alias('distinct_customers'),
    sum(when(col('order_id').isNull(), 1).otherwise(0)).alias('unmatched_orders')
).display()

# Gold Customer

In [0]:
gold_customer_performance = customer_order_details.groupBy(
    'customer_unique_id'
).agg(
    countDistinct('order_id').alias('total_orders'),
    round(sum('total_price'), 2).alias('total_spending'),
    round(avg('total_item_value'), 2).alias('avg_order_value'),
    sum('total_items').alias('total_items_purchased')
)

gold_customer_performance.display()

In [0]:
# validation
gold_customer_performance.agg(
    count('*').alias('total_customers'),
    countDistinct('customer_unique_id').alias('distinct_customers'),
    sum('total_orders').alias('total_orders'),
    sum('total_items_purchased').alias('total_items_purchased'),
    sum(when(col('total_spending').isNull(), 1).otherwise(0)).alias('null_spending'),
    sum(when(col('avg_order_value').isNull(), 1).otherwise(0)).alias('null_avg_order_value')
).display()

In [0]:
# inspecting null values
customer_order_details.filter(
    col('total_item_value').isNull()
).groupBy(
    'order_status'
).count().orderBy(
    col('count').desc()
).display()

In [0]:
# updating previous table
gold_customer_performance = customer_order_details.groupBy(
    'customer_unique_id'
).agg(
    countDistinct('order_id').alias('total_orders'),
    count('total_item_value').alias('orders_with_item_data'),
    sum(when(col('total_item_value').isNull(), 1).otherwise(0))
        .alias('orders_missing_item_data'),
    round(sum('total_price'), 2).alias('total_spending'),
    round(avg('total_item_value'), 2).alias('avg_order_value'),
    sum('total_items').alias('total_items_purchased')
)

gold_customer_performance.display()

In [0]:
gold_customer_performance.agg(
    count('*').alias('total_customers'),
    countDistinct('customer_unique_id').alias('distinct_customers'),
    sum('total_orders').alias('total_orders'),
    sum('orders_with_item_data').alias('orders_with_item_data'),
    sum('orders_missing_item_data').alias('orders_missing_item_data'),
    sum('total_items_purchased').alias('total_items_purchased')
).display()

In [0]:
# saving gold_customer_performence
gold_customer_performance.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.gold_customer_analytics')

In [0]:
%sql
describe detail workspace.olist.gold_customer_analytics

## Seller Analytics

Seller analytics joins `silver_sellers` to `silver_order_items` on `seller_id` with an inner join (`seller_order_details`), connecting every sold item to the seller behind it.

**Per-seller performance** (`gold_seller_performance`, grouped by `seller_id`):

* `total_items_sold` — `count(order_id)`, units the seller sold
* `total_orders` — `countDistinct(order_id)`, the number of distinct orders the seller participated in (a seller can sell several items within one order, so counting rows would overstate orders)
* `total_revenue` — `sum(price)`, the seller's product revenue
* `total_freight` — `sum(freight_value)`, shipping charged on those items
* `total_order_value` — `sum(price) + sum(freight_value)`, the full monetary value
* `avg_selling_price` — `avg(price)`, the seller's average unit price

The seller location columns are attached back with a LEFT JOIN against `silver_sellers`, producing `gold_seller_analytics` (`workspace.olist.gold_seller_analytics`). Seller-level performance shows which sellers drive revenue and volume — the basis for seller management, commission decisions and logistics planning.

In [0]:
# connecting seller to the item they sold
seller_order_details = silver_sellers.join(
    silver_order_items,
    on='seller_id',
    how='inner'
)

In [0]:
seller_order_details.agg(
    count('*').alias('total_rows'),
    countDistinct('seller_id').alias('distinct_sellers'),
    countDistinct('order_id').alias('distinct_orders')
).display()

# Gold Seller

In [0]:
gold_seller_performance = seller_order_details.groupBy('seller_id').agg(
    count('order_id').alias('total_items_sold'),
    countDistinct('order_id').alias('total_orders'),
    round(sum('price'), 2).alias('total_revenue'),
    round(sum('freight_value'), 2).alias('total_freight'),
    round(sum('price') + sum('freight_value'), 2).alias('total_order_value'),
    round(avg('price'), 2).alias('avg_selling_price')
)

In [0]:
gold_seller_performance.agg(
    count('*').alias('total_sellers'),
    countDistinct('seller_id').alias('distinct_sellers')
).display()

In [0]:
gold_seller_analytics = gold_seller_performance.join(
    silver_sellers,
    on='seller_id',
    how='left'
)

In [0]:
gold_seller_analytics.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.gold_seller_analytics')

In [0]:
gold_seller_analytics.agg(
    count('*').alias('total_rows'),
    countDistinct('seller_id').alias('distinct_sellers'),
    sum(when(col('seller_city').isNull(), 1).otherwise(0)).alias('null_cities'),
    sum(when(col('seller_state').isNull(), 1).otherwise(0)).alias('null_states')
).display()

## Seller State Performance

`gold_seller_state_performance` aggregates `seller_order_details` by `seller_state`:

* `total_sellers` — `countDistinct(seller_id)`, the active seller base per state
* `total_items_sold` — `count(order_id)`, units sold through the state's sellers
* `total_orders` — `countDistinct(order_id)`, distinct orders that involved at least one of the state's sellers
* `total_revenue`, `total_freight`, `total_order_value` — the corresponding sums of `price`, `freight_value` and their total

`total_orders` uses `countDistinct` deliberately: an order can contain items from multiple sellers — potentially sellers in different states — so simply summing per-seller order counts would count the same order several times over. Distinct counting measures the unique orders touched by each state's sellers. Persisted as `workspace.olist.gold_seller_state_performance`.

In [0]:
gold_seller_state_performance = seller_order_details.groupBy(
    'seller_state'
).agg(
    countDistinct('seller_id').alias('total_sellers'),
    count('order_id').alias('total_items_sold'),
    countDistinct('order_id').alias('total_orders'),
    round(sum('price'), 2).alias('total_revenue'),
    round(sum('freight_value'), 2).alias('total_freight'),
    round(sum('price') + sum('freight_value'), 2).alias('total_order_value')
)

In [0]:
gold_seller_state_performance.write.format('delta') \
    .mode('overwrite') \
    .saveAsTable('workspace.olist.gold_seller_state_performance')

## Gold Delta Tables and Analytical Outputs

The Gold tables created in this notebook, all in `workspace.olist`:

| Gold table | Grain | Analytical purpose |
| --- | --- | --- |
| `monthly_orders` | year + month | Order volume trend over time |
| `golden_order_summary` | one row per order with items | Per-order item count, price, freight and total value |
| `gold_order_analytics` | one row per order | Full order attributes plus item summary (LEFT JOIN, all orders retained) |
| `gold_product_analytics` | one row per product | Units sold, revenue, freight, order value, average selling price and category |
| `gold_category_performance` | one row per category | Category-level product counts, sales, revenue and pricing |
| `gold_customer_analytics` | one row per person (`customer_unique_id`) | Orders, spending, average order value and items purchased per customer |
| `gold_seller_analytics` | one row per seller | Seller sales, revenue, freight and pricing plus location |
| `gold_seller_state_performance` | one row per seller state | Seller base, volume and revenue per state |

Each table is one flat row per business entity or time period with the metrics already aggregated — the shape dashboards and BI reports consume directly, without recomputing joins and aggregations on every query. The end-to-end validation notebook (`04_validation_optimization_sql`) reconciles the revenue, freight, item and order totals in these tables against `silver_order_items` to confirm the Gold layer faithfully represents the Silver data.

## Gold Delta Tables and Analytical Outputs

The Gold tables created in this notebook, all in `workspace.olist`:

| Gold table | Grain | Analytical purpose |
| --- | --- | --- |
| `monthly_orders` | year + month | Order volume trend over time |
| `golden_order_summary` | one row per order with items | Per-order item count, price, freight and total value |
| `gold_order_analytics` | one row per order | Full order attributes plus item summary (LEFT JOIN, all orders retained) |
| `gold_product_analytics` | one row per product | Units sold, revenue, freight, order value, average selling price and category |
| `gold_category_performance` | one row per category | Category-level product counts, sales, revenue and pricing |
| `gold_customer_analytics` | one row per person (`customer_unique_id`) | Orders, spending, average order value and items purchased per customer |
| `gold_seller_analytics` | one row per seller | Seller sales, revenue, freight and pricing plus location |
| `gold_seller_state_performance` | one row per seller state | Seller base, volume and revenue per state |

Each table is one flat row per business entity or time period with the metrics already aggregated — the shape dashboards and BI reports consume directly, without recomputing joins and aggregations on every query. The end-to-end validation notebook (`04_validation_optimization_sql`) reconciles the revenue, freight, item and order totals in these tables against `silver_order_items` to confirm the Gold layer faithfully represents the Silver data.